<a href="https://colab.research.google.com/github/davesagit123/blank-app/blob/main/algorithmic_tier_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import numpy as np
import pandas as pd
import io

# ==========================================
# 1. SETUP SAMPLE DATA (Paste directly from spreadsheet below)
# ==========================================
# Just paste your rows inside the triple quotes below.
# It automatically handles tab or space separation.
raw_data = """
2	KRAKEN TIME	4.6	9	6.5
1	I'M CRAFTY	6.5	6	5.5
9	EYE OF THE SUN	7.5	9	9.5
5	GOD'S GATE	9.5	4.2	6.5
6	KING'S DRAMA	11	7.5	13
4	FINE STAR	13	20	9.5
3	ON PATROL	16	26	21
11	RETURN OF THE MACH	21	31	26
8	COSMIC TREAT	26	15	17
10	KATONA	31	41	71
13	WOLF WHISTLE ZEL	51	61	41
12	ROK 'N' ROM	71	91	51
7	REGAL SERVICE	101	101	41
"""

# Read raw data using a flexible regex separator (handles both tabs and spaces)
df = pd.read_csv(
    io.StringIO(raw_data.strip()),
    sep=r'\s{2,}|\t',
    engine='python',
    header=None,
    names=['tab', 'horse', 'neural_career', 'current_form', 'auxiliary']
)

# Invert odds format so HIGHER represents better performance
cols_to_invert = ['neural_career', 'current_form', 'auxiliary']
df_inv = df.copy()
for col in cols_to_invert:
  df_inv[col] = 1.0 / df_inv[col]

# ==========================================
# 2. METHOD 1: PARETO / SKYLINE TIERING
# ==========================================

def compute_skyline(df_subset, criteria_cols):
  values = df_subset[criteria_cols].values
  n = len(values)
  is_skyline = np.ones(n, dtype=bool)

  for i in range(n):
    for j in range(n):
      if i != j:
        if np.all(values[j] >= values[i]) and np.any(values[j] > values[i]):
          is_skyline[i] = False
          break
  return is_skyline

def compute_rainbow_layers(df_subset, criteria_cols):
  remaining = df_subset.copy()
  layers = {}
  layer_idx = 1

  while len(remaining) > 0:
    # Use our reliable compute_skyline helper directly on the remaining subset
    is_sky = compute_skyline(remaining, criteria_cols)
    sky_rows = remaining.index[is_sky]

    for r in sky_rows:
      layers[r] = layer_idx

    remaining = remaining.drop(sky_rows)
    layer_idx += 1

  return pd.Series(layers)

df['skyline_tier_1'] = compute_skyline(df_inv, cols_to_invert)
df['rainbow_layer'] = compute_rainbow_layers(df_inv, cols_to_invert)

# ==========================================
# 3. METHOD 2: REFINED COSKY & TOPSIS (Ideal-Point & Distance)
# ==========================================

def compute_refined_cosky_topsis(df_subset, criteria_cols):
  mat = df_subset[criteria_cols].values
  # Column-wise normalization (vector length norm / L2 norm for robust geometric distances)
  norm_factor = np.linalg.norm(mat, axis=0)
  norm_mat = mat / norm_factor

  # Find Ideal (Best) and Anti-Ideal (Worst) solutions
  ideal = norm_mat.max(axis=0)
  anti_ideal = norm_mat.min(axis=0)

  scores = []
  for i in range(len(mat)):
    vec = norm_mat[i]

    # 1. Angle/Correlation component: Cosine similarity to Ideal
    dot_prod = np.dot(vec, ideal)
    norm_vec = np.linalg.norm(vec)
    norm_ideal = np.linalg.norm(ideal)
    cosine_sim = dot_prod / (norm_vec * norm_ideal) if (norm_vec * norm_ideal) > 0 else 0

    # 2. Magnitude component: Euclidean Distance to Ideal (minimize) and Anti-Ideal (maximize)
    dist_to_ideal = np.linalg.norm(vec - ideal)
    dist_to_anti_ideal = np.linalg.norm(vec - anti_ideal)

    # TOPSIS Relative Closeness: higher is closer to ideal and further from worst
    topsis_score = dist_to_anti_ideal / (dist_to_ideal + dist_to_anti_ideal) if (dist_to_ideal + dist_to_anti_ideal) > 0 else 0

    # Combine Cosine Similarity (profile shape) with TOPSIS Score (magnitude metric)
    combined_performance = 0.5 * cosine_sim + 0.5 * topsis_score
    scores.append(combined_performance)

  return scores

df['refined_score'] = compute_refined_cosky_topsis(df_inv, cols_to_invert)
df['refined_rank'] = df['refined_score'].rank(ascending=False).astype(int)

# Sort by overall performance (Rainbow Layer first, then Refined Rank)
df = df.sort_values(by=['rainbow_layer', 'refined_rank']).reset_index(drop=True)

# ==========================================
# 4. DISPLAY COMPACT OUTPUT
# ==========================================
print('--- REFINED HORSE RACING ALGORITHMIC RANKINGS ---')
output_cols = [
    'tab',
    'horse',
    'neural_career',
    'current_form',
    'auxiliary',
    'skyline_tier_1',
    'rainbow_layer',
    'refined_rank'
]
display(df[output_cols])

--- REFINED HORSE RACING ALGORITHMIC RANKINGS ---


,tab,horse,neural_career,current_form,auxiliary,skyline_tier_1,rainbow_layer,refined_rank
0,1,I'M CRAFTY,6.5,6.0,5.5,True,1,1
1,5,GOD'S GATE,9.5,4.2,6.5,True,1,2
2,2,KRAKEN TIME,4.6,9.0,6.5,True,1,3
3,9,EYE OF THE SUN,7.5,9.0,9.5,False,2,4
4,6,KING'S DRAMA,11.0,7.5,13.0,False,2,5
5,4,FINE STAR,13.0,20.0,9.5,False,3,6
6,8,COSMIC TREAT,26.0,15.0,17.0,False,3,7
7,3,ON PATROL,16.0,26.0,21.0,False,4,8
8,11,RETURN OF THE MACH,21.0,31.0,26.0,False,5,9
9,10,KATONA,31.0,41.0,71.0,False,6,10
